# Classical baseline: feature extraction on Colab

The features step needs the 10,480 MILK10k images (about 30 minutes on the free CPU runtime).
This notebook clones the repo, downloads MILK10k from the ISIC Archive (DOI 10.34970/648456, 345 MB),
runs `python classical_baseline.py features` and downloads `outputs/classical_baseline/` as a zip.
The other steps (`compare`, `final`, `stretch`) only need `image_features.csv.gz` and run anywhere:

```
python classical_baseline.py compare final stretch
```

In [ ]:
!rm -rf /content/milk10k-eda && git clone -q https://github.com/bporto-eada/milk10k-eda.git /content/milk10k-eda
%cd /content/milk10k-eda
!pip -q install "scikit-image>=0.21"

In [ ]:
import os, time
from pathlib import Path

if not Path("/content/milk10k.zip").exists():
    !wget -q --show-progress -O /content/milk10k.zip https://isic-archive.s3.amazonaws.com/dois/10.34970-648456/milk10k.zip
!unzip -q -o /content/milk10k.zip -d /content/milk10k_raw

root = Path("/content/milk10k_raw")
jpgs = sorted(root.rglob("*.jpg"))
if len({p.parent for p in jpgs}) == 1:
    image_dir = jpgs[0].parent
else:                                       # one folder per lesion: link every image into one folder
    image_dir = Path("/content/milk10k_images")
    image_dir.mkdir(exist_ok=True)
    for p in jpgs:
        if not (image_dir / p.name).exists():
            (image_dir / p.name).symlink_to(p)
meta = next(root.rglob("metadata.csv"), None)
os.environ["MILK10K_DIR"] = str(meta.parent if meta else root)
os.environ["MILK10K_IMAGES_DIR"] = str(image_dir)
print(len(jpgs), "images in", image_dir)

In [ ]:
start = time.time()
!python -W ignore classical_baseline.py features
print(f"{(time.time() - start) / 60:.1f} min")

In [ ]:
!cd outputs && zip -q -r /content/classical_baseline_features.zip classical_baseline
from google.colab import files
files.download("/content/classical_baseline_features.zip")